# core

> Scheme R5RS implementation in Python

Goal here is to incrementally build up to an implementation of Lisp in python. I'd like to use the solveit method starting from the inside out. So we'll likely start with literals and work our way up.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Great. An inside-out path for a tiny Lisp in Python could start like this:

1. **Data/literals**: numbers, strings, booleans, symbols, lists
2. **Reader/parser**: turn text like `(+ 1 2)` into Python data structures
3. **Environment**: variable lookup and bindings
4. **Evaluator**: evaluate literals, symbols, and function calls
5. **Special forms**: `quote`, `if`, `define`, `lambda`
6. **Functions/macros later**: builtins, closures, maybe macros

For the very first step, I’d separate **self-evaluating literals** from **symbols**.

A minimal representation might be:

```python
class Symbol(str): pass
```

Then plain Python values can represent literals:

```python
42          # number literal
"hello"     # string literal
True        # boolean literal
Symbol("x") # Lisp symbol
```

The key distinction: `"x"` is a string value, but `Symbol("x")` means “look up the variable named x” once evaluation exists.

<details class='token-usage-details' markdown='1'><summary>$0.0048</summary>

`total=30,987 | in=30,751 | out=236 | cached=4.6% | $0.0048 | gpt-5.5`

</details>


In [ ]:
#| default_exp core

In [ ]:
#| export
from fastcore.basics import store_attr, first

In [ ]:
#| export
from compact.reader import *
from compact.types import *

In [ ]:
#| export
def pair2list(p):
    "walk a Pair chain into a python list"
    return list(p)

### The Evaluator: eval and apply

help me understand how eval and apply work together?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

`eval` and `apply` are the two halves of a Lisp interpreter:

- **`eval`** answers: “What does this expression mean in this environment?”
- **`apply`** answers: “Given a function and already-evaluated arguments, how do I call it?”

For a call expression like:

```scheme
(+ 1 2)
```

`eval` does the outer work:

1. sees this is a list
2. evaluates the operator `+` → finds the Python/Lisp function for addition
3. evaluates each argument: `1 → 1`, `2 → 2`
4. hands them to `apply`

Conceptually:

```python
proc = eval(op, env)
args = [eval(arg, env) for arg in expr[1:]]
return apply(proc, args)
```

Then `apply` actually invokes the procedure.

The important split is: **`eval` decides what must be evaluated; `apply` just calls something with values**. Special forms like `if` and `define` are special because they *don’t* evaluate all their arguments before acting.

<details class='token-usage-details' markdown='1'><summary>$0.0049</summary>

`total=31,455 | in=31,219 | out=236 | cached=20.9% | $0.0049 | gpt-5.5`

</details>


In [ ]:
#| export
def scm_apply(fn, args): return fn(*args)

**Tail-Call Optimization**
Naive recursive evaluation stack overflows for deep recursion (e.g. `(fact 10000)`). Instead of recursing, we can evaluate iteratively.`scm_eval_one_step` and `scm_eval_tco` recursively call each other.  

One step returns expression, if it is a thunk, it gets executed in the scm_eval_tco. This avoids having deep stacks.

In [ ]:
#| export
def scm_eval_one_step(expr, env, sfs=()):
    "eval-uate lisp expressions given an environment and handlers for special forms"
    # Atoms
    if isinstance(expr, Symbol): return env[expr.s]
    if not isinstance(expr, Pair): return expr

    # lists
    hd, body = car(expr), pair2list(cdr(expr))

    if isinstance(hd, Symbol) and hd.s in sfs: return sfs[hd.s](body, env, sfs)
    fn = scm_eval_tco(hd, env, sfs)
    if isinstance(fn, Macro): return Thunk(fn.fn(*body), env)

    return scm_apply(fn, [scm_eval_tco(o, env, sfs) for o in body])

In [ ]:
#| export
def scm_eval_tco(expr, env, sfs=()):
    "eval but with tail call optimization"
    while True:
        r = scm_eval_one_step(expr, env, sfs)
        if isinstance(r, Thunk): expr, env = r.expr, r.env
        else: return r

### Environment

In [ ]:
#| export
class Env:
    "this is an implementation of env for scheme eval()"
    def __init__(self, d, primitives:dict=(), parent=None): store_attr(cast=True)

    def __getitem__(self, k):
        s = self.find(k)
        if s is not None: 
            # there could be collisions for keys in primitives and globals()
            # for instance if `list` is called in scheme we want the builtin version
            # in python list will return what is in globals
            # (define list 42) is the moral equivalent of list = 42 - they will overwrite what's in globals
            if k in s.primitives: return s.primitives[k]
            return s.d[k]
        raise KeyError(k)

    def __setitem__(self, k, v): self.d[k] = v

    def __contains__(self, k): return self.find(k) is not None

    def update(self, d): self.primitives.update(d)

    def find(self, k):
        if k in self.d | self.primitives: return self
        if self.parent is not None: return self.parent.find(k)
        return None
    
    def new_frame(self, bindings=()):
        return Env(dict(bindings), parent=self)



### Primitive Procedures
Give the interpreter some basic skills.

In [ ]:
#| export
import math, operator as op

def scm_sub(x, *xs): return x - sum(xs) if xs else -x
def scm_div(x, *xs): return 1/x if not xs else x / math.prod(xs)

def is_num(x): return not isinstance(x, bool) and isinstance(x, (int, float, complex))
def is_sym(x): return isinstance(x, Symbol)

_builtin = {
    "+": lambda *xs: sum(xs),
    "*": lambda *xs: math.prod(xs), 
    "/": scm_div,
    "-": scm_sub,
    ">": op.gt, 
    "<": op.lt, 
    "=": lambda x,y: is_num(x) and is_num(y) and x == y,
    "<=": op.le,
    ">=": op.ge,
    
    "number?": is_num,
    "string?": lambda x: isinstance(x, str),
    "symbol?": is_sym,
    "boolean?": lambda x: isinstance(x, bool),
}

def is_list(x): return x is Nil or (isinstance(x, Pair) and is_list(cdr(x)))
def is_pair(x): return isinstance(x, Pair)

pair_primitives = {
    "list": lambda *xs: list2pair(list(xs)),
    "cons": cons, 
    "car": car, 
    "cdr": cdr, 
    "null?": lambda x: x is Nil, 
    "pair?": is_pair,
    "list?": is_list,
    }

In [ ]:
global_env = Env(globals(), parent=None, primitives=_builtin | pair_primitives)

### Special Forms

The big difference is that arguments for special forms are passed in directly without evaluation. It is up to the the special form to interpret them.

For instance:
```
(if #t 1 (/ 1 0))
```
The `else` clause should not be greedily evaluated. `if` needs to figure out if it wants to handle it.

#### quote
```lisp
    `(+ 1 2) -> (+ 1 2)
```

In [ ]:
#| export
def _sf_quote(xs, env, sfs): return xs[0]

#### if

```lisp
    (if #t 1 (/ 1 0))
```

In [ ]:
#| export
def _sf_if_tco(xs, env, sfs):
    cond,then_,else_ = xs
    br = else_ if scm_eval_tco(cond, env, sfs) is False else then_
    return Thunk(br, env)


#### define — binding symbols

```lisp
    (define name value)

    (define square (lambda (x) (* x x)))

    (define (square x) (* x x))
```

In [ ]:
#| export
def body_expr(body): return body[0] if len(body) == 1 else cons(Symbol("begin"), list2pair(body))

In [ ]:
#| export
def arity(ps, vs):
    ps = list(ps)
    if len(vs) == len(ps): return vs
    raise ValueError(f"incorrect number of vs passed (wanted: {len(ps)}, got: {len(vs)})")

In [ ]:
#| export
def bind_params(ps, vs):
    if isinstance(ps, Symbol): return {ps.s: list(vs)}
    return dict(zip([p.s for p in ps], arity(ps, vs)))

In [ ]:
#| export
def _sf_define_tco(xs, env, sfs):
    def _mkfn(sym, expr): 
        if not isinstance(sym, Symbol): raise SyntaxError(f"{sym} must be a symbol")
        env[sym.s] = scm_eval_tco(expr, env, sfs)
        return sym.s

    arg0, *rest = xs
    if isinstance(arg0, Symbol): return _mkfn(arg0, body_expr(rest))
    if isinstance(arg0, Pair): return _mkfn(car(arg0), cons(Symbol("lambda"), cons(cdr(arg0), list2pair(rest))))

In [ ]:
%%py
scm_eval_tco(
    parse("""
        (begin
          (define (square x) (* x x))
          (square 5))
    """),
    global_env,
    sfs={"define":_sf_define_tco, "lambda":_sf_lambda_tco, "begin":_sf_begin_tco}
)

25

In [ ]:
%%py
scm_eval_tco(
    parse("""
        (begin
          (define (f x)
            (define y (* x 2))
            (+ y 1))
          (f 5))
    """),
    global_env,
    sfs={"define":_sf_define_tco, "lambda":_sf_lambda_tco, "begin":_sf_begin_tco}
)

11

#### lambda — anonymous procedures

```lisp
    (lambda (x) (+ x 2))
```

In [ ]:
#| export
def _sf_lambda_tco(xs, env, sfs): 
    params, *body = xs
    def procedure(*args):
        return Thunk(body_expr(body), env.new_frame(bind_params(params, args)))
    return procedure
    # return lambda *args: Thunk(body_expr(body), Env(**bind_params(params, args), parent=env))

#### begin

```lisp
    (begin
        (define a 2)
        (+ a 2))
```

In [ ]:
#| export
def _sf_begin_tco(xs, env, sfs):
    for o in xs[:-1]: scm_eval_tco(o, env, sfs)
    return Thunk(xs[-1], env)

#### set!

In [ ]:
def _sf_set(xs, env, sfs):
    sym, expr = xs
    if not isinstance(sym, Symbol): raise SyntaxError(f"set! argument {sym} must be a symbol")
    env.find(sym.s)[sym.s] = scm_eval_tco(expr, env, sfs)
    return sym.s

In [ ]:
scm_eval_tco(parse("""
(begin
  (define x 1)
  (set! x 42)
  x)
"""), global_env, sfs={"begin": _sf_begin_tco, "define": _sf_define_tco, "set!": _sf_set})

42

In [ ]:
scm_eval_tco(
    parse("""
        (begin
          (define fact-iter
            (lambda (n acc)
            (if (= n 0) acc (fact-iter (- n 1) (* acc n)))))
          (fact-iter 5 1))
    """), 
    global_env, 
    sfs={"define":_sf_define_tco, "lambda":_sf_lambda_tco, "if":_sf_if_tco, "begin":_sf_begin_tco}
)

120

In [ ]:
scm_eval_tco(
    parse("""

(begin
  (define fact-iter
    (lambda (n acc)
    (if (= n 0) acc (fact-iter (- n 1) (* acc n)))))
  (fact-iter 1000 1))        
  
    """), 
    global_env, 
    sfs={"define":_sf_define_tco, "lambda":_sf_lambda_tco, "if":_sf_if_tco, "begin":_sf_begin_tco}
)

4023872600770937735437024339230039857193748642107146325437999104299385123986290205920442084869694048004799886101971960586316668729948085589013238296699445909974245040870737599188236277271887325197795059509952761208749754624970436014182780946464962910563938874378864873371191810458257836478499770124766328898359557354325131853239584630755574091142624174743493475534286465766116677973966688202912073791438537195882498081268678383745597317461360853795345242215865932019280908782973084313928444032812315586110369768013573042161687476096758713483120254785893207671691324484262361314125087802080002616831510273418279777047846358681701643650241536913982812648102130927612448963599287051149649754199093422215668325720808213331861168115536158365469840467089756029009505376164758477284218896796462449451607653534081989013854424879849599533191017233555566021394503997362807501378376153071277619268490343526252000158885351473316117021039681759215109077880193931781141945452572238655414610628921879602238389714760

### Primitive List Operations

In [ ]:
scm_eval_tco(parse("(car (cdr '(10 20 30)))"), global_env, sfs={"quote":_sf_quote})

20

### let — lexical bindings
```
    (let ((x 40))
      (+ x 2))

```

In [ ]:
def _sf_let(xs, env, sfs):
    binds, *body = xs
    new_env = env.new_frame({name.s: scm_eval_tco(val, env, sfs) for name, val in binds})

    return Thunk(body_expr(body), new_env)

### Macros
```
    (define unless 
      (macro (test body) `(if ,test #f ,body)))

```

In [ ]:
#| export
def _macro(args, env, sfs):
    params, *body = args
    return Macro(lambda *vals: scm_eval_tco(body_expr(body), env.new_frame(bind_params(params, vals)), sfs))

In [ ]:
#| export
def qq(x, env, sfs, depth=1):
    "handle ` , and ,@"
    QQ, UQ, UQS = Symbol("quasiquote"), Symbol("unquote"), Symbol("unquote-splicing")
    def tagged(o, sym): return isinstance(o, Pair) and o.car == sym and isinstance(o.cdr, Pair) and o.cdr.cdr is Nil

    if not isinstance(x, Pair): return x
    if tagged(x, QQ): return cons(QQ, cons(qq(car(cdr(x)), env, sfs, depth+1), Nil))
    if tagged(x, UQ):
        y = car(cdr(x))
        return scm_eval_tco(y, env, sfs) if depth == 1 else cons(UQ, cons(qq(y, env, sfs, depth-1), Nil))
    if tagged(x, UQS):
        if depth == 1: raise SyntaxError("unquote-splicing only valid inside a list")
        return cons(UQS, cons(qq(car(cdr(x)), env, sfs, depth-1), Nil))

    def walk(p):
        if not isinstance(p, Pair): return p
        o, rest = p.car, p.cdr
        if tagged(o, UQS) and depth == 1: return list2pair(pair2list(scm_eval_tco(car(cdr(o)), env, sfs)), walk(rest))
        return cons(qq(o, env, sfs, depth), walk(rest))
    return walk(x)

### Quasiquote
```lisp
`(1 2 ,(+ 1 2) ,@(list 4 5))
;; => (1 2 3 4 5)
```

In [ ]:
#| export
def _sf_quasiquote(args, env, sfs): return qq(args[0], env, sfs)

In [ ]:
scm_eval_tco(
    parse("""
        (let ((x 42) (xs '(1 2 3))) 
        `(a ,x ,@xs z))
        """
    ), global_env,sfs={"quote":_sf_quote, "quasiquote":_sf_quasiquote, "let":_sf_let}
)

compact.types.Pair(car=compact.types.Symbol(s='a'), cdr=compact.types.Pair(car=42, cdr=compact.types.Pair(car=1, cdr=compact.types.Pair(car=2, cdr=compact.types.Pair(car=3, cdr=compact.types.Pair(car=compact.types.Symbol(s='z'), cdr=()))))))

In [ ]:
scm_eval_tco(
    parse(
    """
    (begin (define unless 
      (macro (test body) `(if ,test #f ,body))) 
      (unless (= 1 2) 42))
    """
    ), global_env, sfs={"quasiquote":_sf_quasiquote, "macro":_macro, "define":_sf_define_tco, "if":_sf_if_tco, "begin":_sf_begin_tco}
)

42

In [ ]:
try: parse("'(. 1)")
except SyntaxError: pass

In [ ]:
try: parse("'(1 . 2 3)")
except SyntaxError: pass

In [ ]:
[o for o in globals().keys() if o.startswith("_sf_")]

['_sf_quote',
 '_sf_if_tco',
 '_sf_define_tco',
 '_sf_lambda_tco',
 '_sf_begin_tco',
 '_sf_set',
 '_sf_let',
 '_sf_quasiquote']

In [ ]:
[k for k in globals() if k.startswith("_sf_")]

['_sf_quote',
 '_sf_if_tco',
 '_sf_define_tco',
 '_sf_lambda_tco',
 '_sf_begin_tco',
 '_sf_set',
 '_sf_let',
 '_sf_quasiquote']

In [ ]:
_all_sfs = {
    "begin": _sf_begin_tco,
    "if": _sf_if_tco,
    "define": _sf_define_tco,
    "lambda": _sf_lambda_tco,
    "macro": _macro,
    "set!": _sf_set,
    "let": _sf_let,
    "quote": _sf_quote,
    "quasiquote": _sf_quasiquote,
    
}

In [ ]:

class LispCtx:
    def __init__(self):
        self.env = Env(globals(), primitives=_builtin | pair_primitives)
    
    def __rmatmul__(self, s):
        return scm_eval_tco(parse(s), self.env, _all_sfs)
        
lisp = LispCtx()

In [ ]:
a = 5
"""
(+ a 6)
""" @ lisp

11

wdyt?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

I like it a lot — this is a clean “small Lisp” architecture.

The strongest bits:

- `scm_eval_one_step` handles the core evaluation rules clearly: atoms, special forms, macro expansion, normal application.
- `scm_eval_tco` + `Thunk` is a nice way to make tail-position evaluation iterative without making every function manually loop.
- `Env.new_frame(...)` gives you lexical scope in a very readable way.
- The `LispCtx` / `"""...""" @ lisp` interface is delightful for notebook exploration.

One thing I’d look at carefully:

```python
def _sf_quote(xs, env, sfs): return list2pair(xs[0])
```

That works for quoted lists, but Scheme quote should also handle symbols, numbers, booleans, strings, `Nil`, etc. So eventually `quote` probably wants to return `xs[0]` directly, or recursively convert Python reader lists into your Scheme `Pair` representation only when needed.

Also, this part is interesting:

```python
self.env = Env(globals(), primitives=_builtin | pair_primitives)
```

That makes Python globals visible inside Lisp, as in your `a = 5` example. That’s very convenient for Solveit/notebook hacking, but it means this is more of an embedded Lisp-in-Python playground than a sealed Scheme environment. Which may be exactly what you want.

Overall: the evaluator/apply/special-form split is solid. I’d probably next tighten the boundary between Python `list` as parsed syntax and Scheme `Pair` as runtime list data, because that’s where quote/quasiquote/macros tend to get subtle.

<details class='token-usage-details' markdown='1'><summary>$0.0062</summary>

`total=38,150 | in=37,304 | out=846 | cached=97.1% | reasoning=494 | $0.0062 | gpt-5.5`

</details>


you seem to keep assuming that we are creating a toy example - where is that coming from. I want a functional way to embed lisp code inside python. 

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Fair pushback. That was my bad framing.

It came from me seeing an incremental interpreter notebook and unconsciously mapping it onto the common “toy Lisp interpreter” pattern. But your code is clearly aiming at something different: **a usable embedded Lisp layer inside Python**, where Python objects, globals, notebook state, and Lisp evaluation intentionally interoperate.

So the right framing is not:

> “toy Scheme interpreter”

but rather:

> “a functional Lisp evaluator/runtime embedded in Python, with Scheme-like syntax and semantics where useful.”

That changes what we should optimize for:

- **Interop ergonomics**: `"""(+ a 6)""" @ lisp` is central, not a gimmick.
- **Runtime data boundary**: decide carefully when values are Python values, Lisp `Pair`s, symbols, thunks, macros, etc.
- **Environment policy**: exposing `globals()` is a design feature, not necessarily leakage.
- **Extensibility**: adding primitives/special forms/macros should feel natural from Python.
- **Correctness where it matters**: lexical scope, tail calls, macro expansion, quote/quasiquote behavior.

So I’ll stop calling it “tiny” or “toy” unless we’re explicitly talking about a reduced test case.

<details class='token-usage-details' markdown='1'><summary>$0.0060</summary>

`total=38,173 | in=37,745 | out=428 | cached=98.7% | reasoning=166 | $0.0060 | gpt-5.5`

</details>


why does _sf_quote not return xs[0] now?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Because right now `quote` is doing a **representation conversion**, not just suppressing evaluation.

Your parser uses Python lists as syntax:

```python
'(10 20 30)   # parsed roughly as [10, 20, 30]
```

But your runtime list primitives use Scheme-style `Pair` / `Nil`:

```python
car, cdr, cons
```

So this works:

```scheme
(car '(10 20 30))
```

because `_sf_quote` turns the parsed Python list into a runtime Lisp list:

```python
def _sf_quote(xs, env, sfs): return list2pair(xs[0])
```

If it returned `xs[0]` directly, then quoted lists would stay as Python lists, and `car`/`cdr` would receive the wrong representation.

The subtlety is: this is only obviously right for quoted **lists**. A more complete version would probably be “convert parsed list structure into runtime Lisp data, but leave atoms alone.” Conceptually:

```python
def quote_val(x): return list2pair(x) if isinstance(x, list) else x
```

So the reason is: `quote` currently bridges from **syntax lists** to **runtime Lisp lists**.

<details class='token-usage-details' markdown='1'><summary>$0.0062</summary>

`total=38,690 | in=38,075 | out=615 | cached=97.8% | reasoning=349 | $0.0062 | gpt-5.5`

</details>


which would not be a problem if we represented expressions as pairs


##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Exactly. If the reader represented Lisp source lists as `Pair`/`Nil` from the start, then `quote` could mostly be the simple Scheme rule:

```python
def _sf_quote(xs, env, sfs): return xs[0]
```

Because the quoted expression would already be in the runtime list representation.

Right now you have a split:

- **Python `list`** = parsed syntax / evaluator call structure
- **`Pair`/`Nil`** = runtime Lisp list data

That split is what forces `_sf_quote` to convert.

If expressions were pairs, then the evaluator would also change shape a bit: instead of `hd, *body = expr`, you’d use `car(expr)` and `cdr(expr)` or convert only at eval boundary.

<details class='token-usage-details' markdown='1'><summary>$0.0059</summary>

`total=38,574 | in=38,409 | out=165 | cached=97.0% | $0.0059 | gpt-5.5`

</details>


actually that would be car(xs)


##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Yes — exactly. If the special form receives its arguments as a Lisp list/pair structure, then `xs` is “the argument list”, so quote returns the first argument:

```python
def _sf_quote(xs, env, sfs): return car(xs)
```

I was thinking in the current Python-list convention where `xs[0]` means first unevaluated argument. In the pair-based representation, the equivalent is `car(xs)`.

<details class='token-usage-details' markdown='1'><summary>$0.0059</summary>

`total=38,824 | in=38,638 | out=186 | cached=96.4% | reasoning=87 | $0.0059 | gpt-5.5`

</details>


and that is only because we don't return pairs from parse consistently


##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Yes — that’s the real root cause.

If `parse` consistently returned Scheme data structures — lists as `Pair`/`Nil`, symbols as `Symbol`, literals as themselves — then `quote` would not need to “repair” representation at all:

```python
def _sf_quote(xs, env, sfs): return car(xs)
```

The current awkwardness comes from mixed representation: parsed call syntax is Python `list`, while runtime list values are `Pair`.

<details class='token-usage-details' markdown='1'><summary>$0.0059</summary>

`total=38,906 | in=38,805 | out=101 | cached=96.0% | $0.0059 | gpt-5.5`

</details>


so what is your opposition to doing that?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

No real opposition — I think returning `Pair`/`Nil` consistently from `parse` is probably the cleaner direction for your goal.

The main tradeoff is that it pushes the representation change “downward” into the reader, so the evaluator can no longer casually do Python-list things like:

```python
hd, *body = expr
```

It would instead need to think in Lisp structure:

```python
hd, body = car(expr), cdr(expr)
```

But that seems like a good trade if you want a real embedded Lisp runtime, because then:

- parsed code and runtime list data share one representation
- `quote` becomes simpler
- macros receive and return the same kind of data the evaluator understands
- quasiquote/unquote become more semantically honest
- `cons`, `car`, `cdr`, `list?`, etc. stop living in a half-Python-list / half-Lisp-list world

The only reason to keep Python lists is implementation convenience while sketching the evaluator. But if the aim is a functional Lisp embedded in Python, I’d prefer the consistent `Pair` representation.

<details class='token-usage-details' markdown='1'><summary>$0.0061</summary>

`total=39,304 | in=38,973 | out=331 | cached=98.2% | reasoning=96 | $0.0061 | gpt-5.5`

</details>


ok so what would need to change. parse, both the eval functions and all of the _sf ?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Yes — but the changes are concentrated.

You’d mainly change **three layers**:

1. **`parse` / reader**
   - Lists should come out as `Pair`/`Nil`, not Python `list`.
   - So parsed `(+ 1 2)` becomes roughly `Pair(Symbol("+"), Pair(1, Pair(2, Nil)))`.

2. **`scm_eval_one_step`**
   - This is the biggest core change.
   - Instead of Python-list destructuring, use `car`/`cdr`:

```python
hd, body = car(expr), cdr(expr)
args = [scm_eval_tco(o, env, sfs) for o in pair2list(body)]
```

   `scm_eval_tco` itself probably barely changes, since it just keeps forcing `Thunk`s.

3. **Special forms**
   - Yes, each `_sf_*` currently assumes `xs` is a Python list.
   - They would need to treat `xs` as a Scheme argument list.
   - At first, you can make this painless by doing `xs = pair2list(xs)` at the top of each special form, then later make them more natively pair-based.

Examples:

- `_sf_quote`: becomes `return car(xs)`
- `_sf_if_tco`: gets `cond`, `then_`, `else_` from the pair arg list
- `_sf_lambda_tco`: `params = car(xs)`, body is `cdr(xs)`
- `_sf_begin_tco`: evaluate all but last expression in a pair list
- `_sf_define_tco`, `_sf_let`, `_macro`, `qq`: all need updates because they currently index/unpack Python lists

The nice part is that your **runtime list primitives** already point this direction: `cons`, `car`, `cdr`, `list`, `pair?`, `list?` are already `Pair`/`Nil`-oriented. So this would remove the current split between “syntax lists” and “runtime lists.”

<details class='token-usage-details' markdown='1'><summary>$0.0066</summary>

`total=40,229 | in=39,284 | out=945 | cached=97.4% | reasoning=516 | $0.0066 | gpt-5.5`

</details>


### nbdev postscript

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()